<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_analysis/Market_Distribution_Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# eBay Soccer Card Market Distribution Overview

This notebook analyzes the `eBay_Market_Data` dataset and reproduces the main market-overview figures used in the project README.

The notebook is designed for **Google Colab** and uses only common Python libraries:

- `pandas` for data preparation
- `numpy` for numeric calculations
- `matplotlib` for charts

## Outputs

The notebook creates:

1. **Figure 1** — Seller asking-price distribution
2. **Figure 2** — Cumulative concentration of listings across players
3. **Figure 3** — Players with the highest listing volume
4. **Figure 4** — Listing volume by Topps and Panini box set
5. Summary tables for the main market statistics

All figures are saved at **500 DPI** in a folder named `figure`.

## 1. Import the required libraries

This section loads the Python libraries used throughout the notebook and creates the output folder for the figures.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Create an output folder for the figures.
FIGURE_DIR = Path("figure")
FIGURE_DIR.mkdir(exist_ok=True)

print("Libraries loaded successfully.")
print(f"Figures will be saved to: {FIGURE_DIR.resolve()}")

## 2. Upload the CSV or XLSX file

Run this cell in Google Colab and upload your `eBay_Market_Data` file.

The notebook will automatically use the first uploaded CSV or XLSX file, so it will still work if the uploaded file has a slightly different filename.

In [ ]:
from google.colab import files
import pandas as pd

# Upload file
uploaded = files.upload()

# Find supported files
data_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".csv", ".xlsx"))
]

if not data_files:
    raise FileNotFoundError(
        "No CSV or XLSX file was uploaded."
    )

FILE_PATH = data_files[0]

print(f"Using file: {FILE_PATH}")


# ============================================================
# LOAD DATA
# ============================================================

if FILE_PATH.lower().endswith(".csv"):
    df = pd.read_csv(FILE_PATH)

elif FILE_PATH.lower().endswith(".xlsx"):
    df = pd.read_excel(FILE_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

display(df.head())

## 3. Load and validate the dataset

The analysis uses four main columns:

- `Player_Canonical` — standardized player name
- `asking_price_usd` — seller asking price in U.S. dollars
- `brand` — card manufacturer, mainly Topps or Panini
- `product_line` — product or set family such as Chrome, Prizm, or Select

The validation step stops the notebook early if any required column is missing.

In [ ]:
df = pd.read_csv(FILE_PATH)

required_columns = [
    "Player_Canonical",
    "asking_price_usd",
    "brand",
    "product_line"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The dataset is missing these required columns: "
        + ", ".join(missing_columns)
    )

print("Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

display(df.head())

## 4. Clean the main analysis columns

This step creates cleaned versions of the brand and product-line fields and converts asking price to numeric format.

Rows are not deleted from the original dataframe. Instead, each analysis section filters only the rows it needs.

In [ ]:
df["asking_price_usd"] = pd.to_numeric(
    df["asking_price_usd"],
    errors="coerce"
)

df["brand_clean"] = (
    df["brand"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["product_line_clean"] = (
    df["product_line"]
    .astype("string")
    .str.strip()
)

df["Player_Canonical"] = (
    df["Player_Canonical"]
    .astype("string")
    .str.strip()
)

print("Main analysis columns cleaned.")

# Figure 1 — Seller Asking-Price Distribution

The goal of this section is to show where most seller asking prices are concentrated.

The price bands are:

- Under $25
- $25–49
- $50–99
- $100–249
- $250–499
- $500–999
- $1,000 or more

The chart uses the percentage of listings in each price band rather than the raw listing count, which makes the market distribution easier to interpret.

In [ ]:
# Keep only rows with a valid asking price.
price_df = df[df["asking_price_usd"].notna()].copy()

price_bins = [
    -np.inf,
    25,
    50,
    100,
    250,
    500,
    1000,
    np.inf
]

price_labels = [
    "< $25",
    "$25–49",
    "$50–99",
    "$100–249",
    "$250–499",
    "$500–999",
    "$1,000+"
]

price_df["price_group"] = pd.cut(
    price_df["asking_price_usd"],
    bins=price_bins,
    labels=price_labels,
    right=False
)

price_summary = (
    price_df["price_group"]
    .value_counts(sort=False)
    .rename("listings")
    .reset_index()
)

price_summary["percentage"] = (
    price_summary["listings"]
    / price_summary["listings"].sum()
    * 100
)

display(price_summary)

### Key price statistics

The **mean** can be strongly affected by a small number of very expensive cards.

The **median** is usually a better description of the typical listing because half of the observed listings are below it and half are above it.

In [ ]:
mean_price = price_df["asking_price_usd"].mean()
median_price = price_df["asking_price_usd"].median()

below_100 = (
    price_df["asking_price_usd"].lt(100).mean() * 100
)

above_500 = (
    price_df["asking_price_usd"].ge(500).mean() * 100
)

print(f"Valid price listings : {len(price_df):,}")
print(f"Mean asking price    : ${mean_price:,.2f}")
print(f"Median asking price  : ${median_price:,.2f}")
print(f"Listings below $100  : {below_100:.1f}%")
print(f"Listings at $500+    : {above_500:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

bars = ax.bar(
    price_summary["price_group"].astype(str),
    price_summary["percentage"]
)

# Add the percentage above each bar.
for bar, value in zip(bars, price_summary["percentage"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12
    )

ax.set_title(
    "Distribution of Seller Asking Prices for Soccer Card Listings",
    fontsize=18,
    fontweight="bold",
    loc="left",
    pad=28
)

ax.text(
    0,
    1.01,
    f"High-confidence matched eBay listings (n = {len(price_df):,}); prices in USD",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Seller asking price (USD)", fontsize=12)
ax.set_ylabel("Share of listings (%)", fontsize=12)

ax.tick_params(axis="x", rotation=30)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.94])

output_path = FIGURE_DIR / "figure_01.png"
plt.savefig(output_path, dpi=500, bbox_inches="tight")
plt.show()

print(f"Saved: {output_path}")


# Player Market Distribution

This section examines how listing volume is distributed across players.

A larger number of listings means that a player is **more represented in the observed sample**. It does **not automatically mean stronger buyer demand**.

In [ ]:
player_counts = (
    df["Player_Canonical"]
    .dropna()
    .loc[lambda s: s.ne("")]
    .value_counts()
)

total_listings = int(player_counts.sum())
total_players = int(len(player_counts))

print(f"Total single-player listings: {total_listings:,}")
print(f"Total players: {total_players:,}")

display(
    player_counts
    .head(15)
    .rename("listings")
    .reset_index()
)

# Figure 2 — Cumulative Concentration of Listings Across Players

Players are ranked from the highest to the lowest listing volume.

The cumulative curve answers questions such as:

- What share of all listings comes from the top 10% of players?
- How many players are needed to represent about 80% of all observed listings?

If the curve rises much faster than the diagonal equal-representation line, listings are concentrated among a smaller group of players.

In [ ]:
# Cumulative listing percentage.
listing_cumulative_pct = (
    player_counts.cumsum()
    / total_listings
    * 100
)

# Cumulative player percentage.
player_cumulative_pct = (
    np.arange(1, total_players + 1)
    / total_players
    * 100
)

# Top 10% of players.
top_10_player_count = int(np.ceil(total_players * 0.10))

top_10_listing_share = (
    player_counts.iloc[:top_10_player_count].sum()
    / total_listings
    * 100
)

# First point where cumulative listings reach at least 80%.
players_for_80 = int(
    np.where(listing_cumulative_pct.to_numpy() >= 80)[0][0] + 1
)

player_share_for_80 = (
    players_for_80
    / total_players
    * 100
)

listing_share_at_80 = float(
    listing_cumulative_pct.iloc[players_for_80 - 1]
)

# Players with five listings or fewer.
players_5_or_less = int((player_counts <= 5).sum())

players_5_or_less_pct = (
    players_5_or_less
    / total_players
    * 100
)

print(
    f"Top 10% of players: {top_10_player_count:,} players "
    f"represent {top_10_listing_share:.1f}% of listings"
)

print(
    f"{players_for_80:,} players "
    f"({player_share_for_80:.1f}% of all players) "
    f"represent {listing_share_at_80:.1f}% of listings"
)

print(
    f"Players with 5 listings or fewer: "
    f"{players_5_or_less:,} "
    f"({players_5_or_less_pct:.1f}% of players)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# Actual concentration curve.
ax.plot(
    player_cumulative_pct,
    listing_cumulative_pct,
    linewidth=3,
    label="Observed listing concentration"
)

# Equal-representation reference line.
ax.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    linewidth=2,
    alpha=0.7,
    label="Equal representation"
)

# Top 10% point.
x_10 = top_10_player_count / total_players * 100
y_10 = top_10_listing_share

ax.scatter(x_10, y_10, s=140, zorder=5)
ax.axvline(x_10, ymax=y_10 / 100, linestyle="--", alpha=0.7)
ax.axhline(y_10, xmax=x_10 / 100, linestyle="--", alpha=0.7)

ax.annotate(
    f"Top 10% of players\n≈ {y_10:.1f}% of listings",
    xy=(x_10, y_10),
    xytext=(25, 69),
    arrowprops=dict(arrowstyle="->"),
    fontsize=12
)

# Approximately 80% of listings point.
x_80 = player_share_for_80
y_80 = listing_share_at_80

ax.scatter(x_80, y_80, s=140, zorder=5)
ax.axvline(x_80, ymax=y_80 / 100, linestyle="--", alpha=0.7)
ax.axhline(y_80, xmax=x_80 / 100, linestyle="--", alpha=0.7)

ax.annotate(
    f"{players_for_80} players\n"
    f"≈ {x_80:.1f}% of players\n"
    f"≈ {y_80:.1f}% of listings",
    xy=(x_80, y_80),
    xytext=(43, 83),
    arrowprops=dict(arrowstyle="->"),
    fontsize=12
)

ax.set_title(
    "Cumulative Concentration of Observed Listings Across Players",
    fontsize=18,
    fontweight="bold",
    loc="left",
    pad=28
)

ax.text(
    0,
    1.01,
    f"Players ranked from highest to lowest listing volume; n = {total_players:,} players",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Cumulative share of players (%)", fontsize=12)
ax.set_ylabel("Cumulative share of listings (%)", fontsize=12)

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.94])

output_path = FIGURE_DIR / "figure_02.png"
plt.savefig(output_path, dpi=500, bbox_inches="tight")
plt.show()

print(f"Saved: {output_path}")


# Figure 3 — Players with the Highest Listing Volume

This chart shows the 15 players with the largest number of observed listings.

Again, listing count should be interpreted as **market representation or data availability**, not direct evidence of buyer demand.

In [ ]:
top_players = (
    player_counts
    .head(15)
    .sort_values()
)

top_players_table = (
    player_counts
    .head(15)
    .rename("listings")
    .reset_index()
)

display(top_players_table)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

bars = ax.barh(
    top_players.index,
    top_players.values
)

for bar, value in zip(bars, top_players.values):
    ax.text(
        value + max(top_players.values) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=11
    )

ax.set_title(
    "Players with the Highest Listing Volume in the Observed eBay Sample",
    fontsize=18,
    fontweight="bold",
    loc="left",
    pad=28
)

ax.text(
    0,
    1.01,
    f"High-confidence single-player listings; n = {total_listings:,}",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Number of observed listings", fontsize=12)
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.94])

output_path = FIGURE_DIR / "figure_03.png"
plt.savefig(output_path, dpi=500, bbox_inches="tight")
plt.show()

print(f"Saved: {output_path}")


# Figure 4 — Brand and Boxset Market Overview

This section compares the main observed boxset from **Topps** and **Panini**.

The product-line lists are fixed to match the boxset used in the project figure. This makes the notebook reproducible even if the CSV contains additional boxset.

Both panels use the **same y-axis scale**, so their listing volumes can be compared directly.

In [ ]:
TOPPS_LINES = [
    "Chrome",
    "Finest",
    "Merlin",
    "Stadium Club Chrome",
    "Living Set",
    "Jade Edition",
    "Team Set",
    "Inception",
    "Topps Now",
    "Crystal"
]

PANINI_LINES = [
    "Prizm",
    "Select",
    "Obsidian",
    "Immaculate",
    "Donruss",
    "Chronicles",
    "Score",
    "Mosaic",
    "Revolution",
    "Adrenalyn XL"
]

topps_counts = (
    df[
        (df["brand_clean"] == "topps")
        & (df["product_line_clean"].isin(TOPPS_LINES))
    ]["product_line_clean"]
    .value_counts()
    .reindex(TOPPS_LINES)
    .fillna(0)
    .astype(int)
)

panini_counts = (
    df[
        (df["brand_clean"] == "panini")
        & (df["product_line_clean"].isin(PANINI_LINES))
    ]["product_line_clean"]
    .value_counts()
    .reindex(PANINI_LINES)
    .fillna(0)
    .astype(int)
)

print("Topps product-line counts")
display(
    topps_counts
    .rename("listings")
    .reset_index()
)

print("Panini product-line counts")
display(
    panini_counts
    .rename("listings")
    .reset_index()
)

### Brand share within the selected boxset

This comparison uses only the boxset included in Figure 4. It should therefore be described as the share **among the selected boxset shown**, not necessarily the complete Topps-versus-Panini market share.

In [ ]:
topps_total = int(topps_counts.sum())
panini_total = int(panini_counts.sum())

brand_total = topps_total + panini_total

topps_share = (
    topps_total / brand_total * 100
    if brand_total else np.nan
)

panini_share = (
    panini_total / brand_total * 100
    if brand_total else np.nan
)

print(f"Topps listings shown : {topps_total:,}")
print(f"Panini listings shown: {panini_total:,}")
print()
print(f"Topps share shown : {topps_share:.1f}%")
print(f"Panini share shown: {panini_share:.1f}%")

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 13)
)

fig.suptitle(
    "Market Composition: Listing Volume by Boxset",
    fontsize=22,
    fontweight="bold",
    y=0.99
)

# Brand colors requested by the user.
TOPPS_COLOR = "#D71921"
PANINI_COLOR = "#FFF442"

# Use one common y-axis limit for both brands.
max_value = max(
    topps_counts.max(),
    panini_counts.max()
)

y_max = max_value * 1.10

# -----------------------------
# Topps
# -----------------------------
bars1 = axes[0].bar(
    topps_counts.index,
    topps_counts.values,
    color=TOPPS_COLOR,
    edgecolor="black"
)

axes[0].set_title(
    "Topps: Top 10 Boxset by Volume",
    fontsize=16,
    fontweight="bold"
)

axes[0].set_ylabel("Number of Listings (Count)")
axes[0].set_xlabel("Boxset")
axes[0].set_ylim(0, y_max)
axes[0].tick_params(axis="x", rotation=45)

for bar, value in zip(bars1, topps_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        value + y_max * 0.008,
        f"{value:,}",
        ha="center",
        fontweight="bold"
    )

# -----------------------------
# Panini
# -----------------------------
bars2 = axes[1].bar(
    panini_counts.index,
    panini_counts.values,
    color=PANINI_COLOR,
    edgecolor="black"
)

axes[1].set_title(
    "Panini: Top 10 Boxset by Volume",
    fontsize=16,
    fontweight="bold"
)

axes[1].set_ylabel("Number of Listings (Count)")
axes[1].set_xlabel("Boxset")
axes[1].set_ylim(0, y_max)
axes[1].tick_params(axis="x", rotation=45)

for bar, value in zip(bars2, panini_counts.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + y_max * 0.008,
        f"{value:,}",
        ha="center",
        fontweight="bold"
    )

# Clean formatting for both panels.
for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])

output_path = FIGURE_DIR / "figure_04.png"
plt.savefig(output_path, dpi=500, bbox_inches="tight")
plt.show()

print(f"Saved: {output_path}")


# Final Summary Tables

These tables collect the main numbers used in the README.

The first table summarizes the overall market and player concentration.

The second table contains the selected Topps and Panini product-line listing counts.

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total single-player listings",
        "Total players",
        "Mean asking price",
        "Median asking price",
        "Listings below $100",
        "Listings at $500 or more",
        "Top 10% player listing share",
        "Players needed to reach about 80% of listings",
        "Share of players needed to reach about 80% of listings",
        "Players with 5 listings or fewer"
    ],
    "Value": [
        f"{total_listings:,}",
        f"{total_players:,}",
        f"${mean_price:,.2f}",
        f"${median_price:,.2f}",
        f"{below_100:.1f}%",
        f"{above_500:.1f}%",
        f"{top_10_listing_share:.1f}%",
        f"{players_for_80:,}",
        f"{player_share_for_80:.1f}%",
        f"{players_5_or_less:,} ({players_5_or_less_pct:.1f}%)"
    ]
})

display(summary)

In [ ]:
product_summary = pd.concat(
    [
        (
            topps_counts
            .rename_axis("product_line")
            .reset_index(name="listings")
            .assign(brand="Topps")
        ),
        (
            panini_counts
            .rename_axis("product_line")
            .reset_index(name="listings")
            .assign(brand="Panini")
        )
    ],
    ignore_index=True
)

product_summary = product_summary[
    ["brand", "product_line", "listings"]
]

display(product_summary)

## Export the summary tables

This optional cell saves the two summary tables as CSV files so they can be used in GitHub, Excel, or another analysis notebook.

In [ ]:
summary.to_csv(
    "market_summary.csv",
    index=False
)

product_summary.to_csv(
    "product_line_summary.csv",
    index=False
)

top_players_table.to_csv(
    "top_15_players.csv",
    index=False
)

print("Saved:")
print("- market_summary.csv")
print("- product_line_summary.csv")
print("- top_15_players.csv")

## Download all generated outputs

This optional final cell creates one ZIP file containing:

- `figure_01.png`
- `figure_02.png`
- `figure_03.png`
- `figure_04.png`
- `market_summary.csv`
- `product_line_summary.csv`
- `top_15_players.csv`

Run the cell to download the ZIP file from Google Colab.

In [ ]:
import shutil
from google.colab import files

OUTPUT_FOLDER = Path("analysis_output")
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Copy figures.
for figure_file in FIGURE_DIR.glob("*.png"):
    shutil.copy(figure_file, OUTPUT_FOLDER / figure_file.name)

# Copy summary tables.
for csv_file in [
    "market_summary.csv",
    "product_line_summary.csv",
    "top_15_players.csv"
]:
    shutil.copy(csv_file, OUTPUT_FOLDER / csv_file)

zip_path = shutil.make_archive(
    "ebay_market_analysis_output",
    "zip",
    OUTPUT_FOLDER
)

print(f"Created: {zip_path}")

files.download(zip_path)